In [4]:
import pandas as pd
import numpy as np

def construct_variables(df):
# Builds the demographic and financial variables needed for the claiming age regression.
# Extracts gender, claiming age, and calculates wealth quartiles
    
    # Build the demographic and financial variables
    # Gender is stored as text (e.g. "1.male"/"2.female")
    # Flag on the substring rather than assuming a fixed numeric code to prevent errors.
    
    df['female'] = df['ragender'].astype(str).str.lower().str.contains('female').astype(int)
    
    # Initialize claim_age as NaN. We will fill it by checking each survey wave.
    df['claim_age'] = np.nan
    
    # Loop through all 16 waves to find the exact age they stopped working
    for w in range(1, 17):
        age_col = f'r{w}agey_e'
        work_col = f'r{w}work'
        
        # Skip if the column doesn't exist in this specific wave
        if age_col not in df.columns or work_col not in df.columns:
            continue
            
        # Define conditions: age must be between 62-70, and they must be not working
        in_range = (df[age_col] >= 62) & (df[age_col] <= 70)
        not_working = df[work_col].astype(str).str.lower().str.contains('not working', na=False)
        
        # still_missing ensures we only take the *first* qualifying wave (their actual retirement/claim age)
        still_missing = df['claim_age'].isna()  # only take the first qualifying wave
        df.loc[still_missing & not_working & in_range, 'claim_age'] = df[age_col].round()

    # Create a binary dummy variable specifically for early claimers (exactly 62)
    df['claimed_at_62'] = (df['claim_age'] == 62).astype(float)

    # Calculate average household wealth across all available waves
    wealth_cols = [c for c in df.columns if 'atotb' in c]
    df['avg_wealth'] = df[wealth_cols].mean(axis=1)

    # Group respondents into 4 wealth quartiles for the bar chart
    df['wealth_quartile'] = pd.qcut(
        df['avg_wealth'].dropna(), q=4,
        labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
    )
    return df


# Load the data passed from the 00_pull notebook
df_raw = pd.read_csv('../output/00_pulled_data.csv')
print("Shape before variable construction:", df_raw.shape)

# Apply the variable construction function
df_clean = construct_variables(df_raw)

# Diagnostic checks (Required by project rubric)
print("Shape after variable construction:", df_clean.shape)
print("Respondents with a non-missing claim_age:", df_clean['claim_age'].notna().sum())
print(df_clean[['female', 'claim_age', 'avg_wealth']].describe())

# Save the cleaned data to be used in the final analysis notebook
df_clean.to_csv('../output/01_merged_data.csv', index=False)
print("Saved to output/01_merged_data.csv")

/var/folders/x3/z35rtwhs2j7gt3thbjwqvcx80000gn/T/ipykernel_72496/3744660422.py:32: DtypeWarning: Columns (6,7,11,12,16,17,21,22,26,27,31,32,36,37,41,42,46,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('../output/00_pulled_data.csv')


Shape before variable construction: (45234, 86)
Shape after variable construction: (45234, 91)
Respondents with a non-missing claim_age: 18241
             female     claim_age    avg_wealth
count  45234.000000  18241.000000  4.523400e+04
mean       0.560707     64.476235  3.613412e+05
std        0.496307      2.561823  1.165708e+06
min        0.000000     62.000000 -1.444645e+06
25%        0.000000     62.000000  2.908195e+04
50%        1.000000     63.000000  1.260000e+05
75%        1.000000     66.000000  3.603694e+05
max        1.000000     70.000000  1.545055e+08
Saved to output/01_merged_data.csv
